[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nahuelcosta/generative-networks/blob/main/reduction/quantization.ipynb)

This notebook is part of the [Generative Networks course](https://www.nahuelcosta.com/teaching/generative-networks).

# Quantizing a Causal Language Model

Quantization maps high-precision values onto a finite set of representable levels. In this tutorial we connect the equation to an actual model tensor, then benchmark the 1.7B-parameter SmolLM2 model in three formats:

- float16;
- LLM.int8;
- NF4 4-bit weight-only quantization.

For every supported format we use the same validation blocks and generation prompt. We record theoretical weight storage, reported model footprint, peak GPU allocation, perplexity, output, and warmed-up generation throughput. Smaller storage does not guarantee proportionally lower latency.

## 0. Dependencies

The repository's local `transformers/` directory makes module-spec checks ambiguous, so this cell checks installed distribution metadata. It pins the validated libraries but preserves Colab's CUDA-enabled PyTorch build.

In [ ]:
from importlib import metadata
import subprocess
import sys

PINNED_PACKAGES = {
    "transformers": "5.14.1",
    "datasets": "5.0.0",
    "accelerate": "1.14.0",
    "bitsandbytes": "0.49.2",
}

to_install = []
for distribution, expected_version in PINNED_PACKAGES.items():
    try:
        installed_version = metadata.version(distribution)
    except metadata.PackageNotFoundError:
        installed_version = None
    if installed_version != expected_version:
        to_install.append(f"{distribution}=={expected_version}")

if to_install:
    print("Installing the validated course environment:", ", ".join(to_install))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *to_install])
else:
    print("Validated dependencies are already installed.")

for distribution, expected_version in PINNED_PACKAGES.items():
    actual_version = metadata.version(distribution)
    print(f"{distribution}: {actual_version}")
    assert actual_version == expected_version

## 1. Imports and Reproducibility

In [ ]:
import gc
import math
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DefaultDataCollator,
    set_seed,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    compute_capability = torch.cuda.get_device_capability(0)
    capability_number = compute_capability[0] * 10 + compute_capability[1]
    print(f"GPU: {gpu_name}; compute capability: {compute_capability}")
else:
    capability_number = 0
    print("CUDA is unavailable. The manual and QAT sections will run; GPU benchmarks will be skipped.")

## 2. Configuration

The complete benchmark requires a CUDA GPU. NF4 requires Pascal-class hardware or newer, while the LLM.int8 path used here requires Turing-class hardware such as a T4. `FAST_DEV_RUN=1` reduces validation, generation, and repetitions without changing formats.

In [ ]:
fast_dev_run = os.environ.get("FAST_DEV_RUN", "0") == "1"

MODEL_ID = "HuggingFaceTB/SmolLM2-1.7B"
CPU_MANUAL_MODEL_ID = "HuggingFaceTB/SmolLM2-135M"
DATASET_ID = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"

project_root = Path.cwd().parent if Path.cwd().name == "reduction" else Path.cwd()
output_dir = project_root / "reduction" / "outputs" / "quantization"
output_dir.mkdir(parents=True, exist_ok=True)

sequence_length = 128 if fast_dev_run else 256
validation_blocks = 4 if fast_dev_run else 32
evaluation_batches = 1 if fast_dev_run else 8
batch_size = 1 if fast_dev_run else 2
generation_tokens = 8 if fast_dev_run else 32
benchmark_repeats = 1 if fast_dev_run else 3

supports_nf4 = capability_number >= 60
supports_llm_int8 = capability_number >= 75
print("FAST_DEV_RUN:", fast_dev_run)
print("LLM.int8 supported:", supports_llm_int8)
print("NF4 supported:", supports_nf4)

## 3. Fixed Evaluation Data

Perplexity is evaluated on the same packed WikiText-2 validation prefix for every format. This small subset is a diagnostic, not a benchmark-suite score.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

raw_validation = load_dataset(
    DATASET_ID, DATASET_CONFIG, split="validation"
)
target_tokens = validation_blocks * sequence_length
token_ids = []
for row in raw_validation:
    text = row["text"].strip()
    if not text:
        continue
    token_ids.extend(tokenizer(text, add_special_tokens=False)["input_ids"])
    token_ids.append(tokenizer.eos_token_id)
    if len(token_ids) >= target_tokens:
        break

token_ids = token_ids[:target_tokens]
blocks = [
    token_ids[start:start + sequence_length]
    for start in range(0, target_tokens, sequence_length)
]
validation_dataset = Dataset.from_dict({
    "input_ids": blocks,
    "attention_mask": [[1] * sequence_length for _ in blocks],
    "labels": [block.copy() for block in blocks],
})
collator = DefaultDataCollator()
assert len(validation_dataset) == validation_blocks
print(validation_dataset)

## 4. Shared Measurement Helpers

Each model is loaded, measured, and released before the next format is loaded. CUDA synchronization prevents asynchronous kernels from producing misleading timings. We report the median rather than assuming a single timing is stable.

In [ ]:
FIXED_PROMPT = "Quantization makes language models"

def parameter_count(model):
    return sum(parameter.numel() for parameter in model.parameters())

@torch.no_grad()
def evaluate_loss(model):
    model.eval()
    model_device = model.get_input_embeddings().weight.device
    loader = DataLoader(
        validation_dataset, batch_size=batch_size, shuffle=False, collate_fn=collator
    )
    losses = []
    for batch_index, batch in enumerate(loader):
        if batch_index >= evaluation_batches:
            break
        batch = {name: value.to(model_device) for name, value in batch.items()}
        losses.append(model(**batch).loss.float().item())
    return float(np.mean(losses))

@torch.no_grad()
def benchmark_generation(model):
    model.eval()
    model_device = model.get_input_embeddings().weight.device
    inputs = tokenizer(FIXED_PROMPT, return_tensors="pt").to(model_device)
    kwargs = {
        "max_new_tokens": generation_tokens,
        "do_sample": False,
        "pad_token_id": tokenizer.eos_token_id,
    }
    _ = model.generate(**inputs, max_new_tokens=2, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    timings = []
    output = None
    for _ in range(benchmark_repeats):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.perf_counter()
        output = model.generate(**inputs, **kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        timings.append(time.perf_counter() - start)

    generated_count = output.size(1) - inputs["input_ids"].size(1)
    return (
        tokenizer.decode(output[0], skip_special_tokens=True),
        generated_count / float(np.median(timings)),
    )

def profile_model(format_name, model, nominal_bits):
    loss = evaluate_loss(model)
    generated_text, tokens_per_second = benchmark_generation(model)
    parameters = logical_parameter_count
    peak_gpu_mb = (
        torch.cuda.max_memory_allocated() / 1e6 if torch.cuda.is_available() else np.nan
    )
    row = {
        "format": format_name,
        "parameters_b": parameters / 1e9,
        "nominal_bits": nominal_bits,
        "theoretical_weight_mb": parameters * nominal_bits / 8 / 1e6,
        "reported_footprint_mb": model.get_memory_footprint() / 1e6,
        "peak_gpu_mb": peak_gpu_mb,
        "validation_loss": loss,
        "perplexity": math.exp(min(loss, 20)),
        "tokens_per_second": tokens_per_second,
        "sample": generated_text,
    }
    print(format_name, {key: value for key, value in row.items() if key != "sample"})
    print(generated_text)
    return row

def release_model(model):
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return None

## 5. Load the Float Model and Select a Real Weight Tensor

On CUDA, this is also the float16 benchmark model. Without CUDA, we load the 135M checkpoint on CPU only for the inspectable quantization exercise and skip the 1.7B benchmark.

In [ ]:
results = []

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    float_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=torch.float16, device_map={"": 0}
    )
else:
    float_model = AutoModelForCausalLM.from_pretrained(
        CPU_MANUAL_MODEL_ID, dtype=torch.float32
    ).to(device)

selected_name = None
selected_weight = None
for name, module in float_model.named_modules():
    if name.endswith("q_proj") and hasattr(module, "weight"):
        selected_name = name + ".weight"
        selected_weight = module.weight.detach().float().cpu()[:256, :256].clone()
        break

logical_parameter_count = parameter_count(float_model)
assert selected_weight is not None
print(f"Selected {selected_name} with slice shape {tuple(selected_weight.shape)}")

if torch.cuda.is_available():
    results.append(profile_model("FP16", float_model, nominal_bits=16))

## 6. Affine Quantization by Hand

For real value `x`, scale `s`, and zero point `z`, affine quantization uses:

`q = clip(round(x / s) + z, q_min, q_max)`

`x_hat = s * (q - z)`

Per-tensor quantization shares one scale across the matrix. Per-channel quantization gives each output row its own scale, allowing narrow-range rows to use the available integer levels more effectively.

In [ ]:
def affine_quantize(tensor, bits, channel_axis=None):
    q_min, q_max = 0, 2**bits - 1
    if channel_axis is None:
        minimum, maximum = tensor.min(), tensor.max()
    else:
        reduction_dimensions = tuple(
            dimension for dimension in range(tensor.ndim)
            if dimension != channel_axis
        )
        minimum = tensor.amin(dim=reduction_dimensions, keepdim=True)
        maximum = tensor.amax(dim=reduction_dimensions, keepdim=True)

    scale = (maximum - minimum) / (q_max - q_min)
    scale = scale.clamp_min(torch.finfo(torch.float32).eps)
    zero_point = torch.round(q_min - minimum / scale).clamp(q_min, q_max)
    quantized = torch.round(tensor / scale + zero_point).clamp(q_min, q_max)
    dequantized = scale * (quantized - zero_point)
    return quantized.to(torch.int16), dequantized, scale, zero_point

def quantization_metrics(original, reconstructed):
    error = original - reconstructed
    return {
        "mse": error.square().mean().item(),
        "maximum_absolute_error": error.abs().max().item(),
        "signal_to_noise_db": 10 * torch.log10(
            original.square().mean() / error.square().mean()
        ).item(),
    }

q8_tensor, dq8_tensor, scale8, zero8 = affine_quantize(selected_weight, 8)
q8_channel, dq8_channel, scale8_channel, zero8_channel = affine_quantize(
    selected_weight, 8, channel_axis=0
)
q4_channel, dq4_channel, scale4_channel, zero4_channel = affine_quantize(
    selected_weight, 4, channel_axis=0
)

manual_results = pd.DataFrame([
    {"scheme": "INT8 per tensor", **quantization_metrics(selected_weight, dq8_tensor)},
    {"scheme": "INT8 per channel", **quantization_metrics(selected_weight, dq8_channel)},
    {"scheme": "4-bit per channel", **quantization_metrics(selected_weight, dq4_channel)},
]).set_index("scheme")
display(manual_results)

assert q8_tensor.min() >= 0 and q8_tensor.max() <= 255
assert q4_channel.min() >= 0 and q4_channel.max() <= 15
assert manual_results.loc["INT8 per channel", "mse"] <= manual_results.loc["INT8 per tensor", "mse"] + 1e-12
assert manual_results.loc["INT8 per channel", "mse"] < manual_results.loc["4-bit per channel", "mse"]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(selected_weight.flatten().numpy(), bins=60)
axes[0].set_title("Original weights")
axes[1].hist(dq4_channel.flatten().numpy(), bins=60)
axes[1].set_title("Dequantized 4-bit weights")
axes[2].hist((selected_weight - dq4_channel).flatten().numpy(), bins=60)
axes[2].set_title("4-bit quantization error")
for axis in axes:
    axis.set_xlabel("Value")
plt.tight_layout()

## 7. Quantization-Aware Training Intuition

`round` has zero gradient almost everywhere, so ordinary backpropagation cannot learn through it. QAT uses fake-quantized values in the forward pass and a straight-through estimator (STE) in the backward pass. Real QAT also learns or observes ranges across a complete model; this tiny example isolates the key gradient trick.

In [ ]:
qat_weights = torch.tensor([-0.63, -0.12, 0.28, 0.91], requires_grad=True)
rounded = torch.round(qat_weights * 4) / 4
rounded.sum().backward()
naive_gradient = qat_weights.grad.clone()

qat_weights.grad.zero_()
quantized_forward = torch.round(qat_weights * 4) / 4
ste_values = qat_weights + (quantized_forward - qat_weights).detach()
ste_values.sum().backward()
ste_gradient = qat_weights.grad.clone()

print("Forward fake-quantized values:", ste_values.detach().tolist())
print("Gradient through round:         ", naive_gradient.tolist())
print("Gradient through STE:           ", ste_gradient.tolist())
assert torch.all(naive_gradient == 0)
assert torch.all(ste_gradient == 1)

## 8. Release the Float Model

The next formats are loaded one at a time. Clearing Python references and the CUDA allocator cache avoids accidentally measuring multiple resident models.

In [ ]:
float_model = release_model(float_model)
print("Float model released.")

## 9. LLM.int8 Benchmark

LLM.int8 keeps outlier computations at higher precision instead of applying naive uniform INT8 to every operation. The cell is skipped on GPUs below compute capability 7.5.

In [ ]:
if supports_llm_int8:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    int8_config = BitsAndBytesConfig(load_in_8bit=True)
    int8_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=int8_config,
        device_map={"": 0},
        dtype=torch.float16,
    )
    results.append(profile_model("LLM.int8", int8_model, nominal_bits=8))
    int8_model = release_model(int8_model)
else:
    print("Skipping LLM.int8: use a T4-class (compute capability 7.5) or newer GPU.")

## 10. NF4 4-Bit Benchmark

NF4 places representable levels to suit normally distributed neural-network weights. Double quantization also compresses quantization constants. Activations and accumulation still use higher precision, so this is not four-bit arithmetic everywhere.

In [ ]:
if supports_nf4:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    nf4_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    nf4_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=nf4_config,
        device_map={"": 0},
        dtype=torch.float16,
    )
    results.append(profile_model("NF4 4-bit", nf4_model, nominal_bits=4))
    nf4_model = release_model(nf4_model)
else:
    print("Skipping NF4: enable a Pascal-class CUDA GPU or newer.")

## 11. Compare Footprint, Quality, and Speed

The theoretical column counts only `parameters * nominal bits`. The reported and peak-memory columns include additional structures and runtime effects. Throughput is deliberately not asserted to improve: kernel support, memory bandwidth, batch size, and dequantization overhead all matter.

In [ ]:
if results:
    results_table = pd.DataFrame(results).set_index("format")
    display(results_table.drop(columns="sample").round(3))

    assert np.isfinite(results_table["validation_loss"]).all()
    if {"FP16", "LLM.int8", "NF4 4-bit"}.issubset(results_table.index):
        assert (
            results_table.loc["FP16", "reported_footprint_mb"]
            > results_table.loc["LLM.int8", "reported_footprint_mb"]
            > results_table.loc["NF4 4-bit", "reported_footprint_mb"]
        )

    metric_columns = [
        "theoretical_weight_mb",
        "reported_footprint_mb",
        "peak_gpu_mb",
        "perplexity",
        "tokens_per_second",
    ]
    axes = results_table[metric_columns].plot(
        kind="bar", subplots=True, layout=(2, 3), figsize=(15, 8), legend=False
    )
    for axis in np.asarray(axes).flat:
        axis.tick_params(axis="x", rotation=20)
    plt.tight_layout()

    print("DETERMINISTIC SAMPLES")
    for format_name, row in results_table.iterrows():
        print(f"\n[{format_name}]\n{row['sample']}")
else:
    print("No GPU benchmark was run. Enable a CUDA runtime for the comparison table.")

## 12. Interpreting the Result

- **Storage:** scales and quantization metadata explain why reported footprint differs from the simple bit-count estimate.
- **Quality:** PTQ introduces numerical error without retraining; whether that error matters must be measured on the target task.
- **Speed:** smaller weights reduce memory traffic, but unsupported or dequantization-heavy kernels can erase that advantage.
- **Compute dtype:** 4-bit storage does not mean 4-bit accumulation; this notebook uses float16 compute for NF4.
- **QAT:** can recover quality at aggressive precision, but requires a new training phase and is outside this PTQ benchmark.

## 13. Things to Try

- Repeat the benchmark with longer prompts and different batch sizes.
- Compare per-tensor, per-channel, and small-group affine quantization on several layers.
- Disable NF4 double quantization and inspect the footprint difference.
- Sweep the LLM.int8 outlier threshold and evaluate quality.
- Quantize the KV cache for long-context generation and separate weight memory from cache memory.
- Compare bitsandbytes PTQ with GPTQ or AWQ using the same evaluation protocol.

## Troubleshooting

- If bitsandbytes cannot find a backend, confirm that Colab is using a GPU runtime and inspect the compute capability printed above.
- If the 1.7B model download fails, restart after checking free disk space; model files are cached between format loads.
- If memory is exhausted, reduce evaluation batch size. Loading formats sequentially is required.
- Do not compare asynchronous CUDA timings without warm-up and synchronization.
- A 4-bit model is not guaranteed to be four times faster than float16; report the measured result for the actual hardware.

**References:** [Bitsandbytes quantization](https://huggingface.co/docs/transformers/quantization/bitsandbytes), [LLM.int8](https://arxiv.org/abs/2208.07339), [QLoRA/NF4](https://arxiv.org/abs/2305.14314), [Quantization-Aware Training](https://arxiv.org/abs/1712.05877), and [WikiText](https://huggingface.co/datasets/Salesforce/wikitext).